# Experiment 1: Case Study Framing & Dataset Preparation

## Aim
To define a real-world loan application approval problem, study existing approaches, acquire and prepare the LendingClub dataset, define success metrics, and version the raw project dataset using Git and DVC.

## Project Title
**Explainable Machine Learning System for Loan Application Approval Prediction**

## Dataset
The project uses two LendingClub datasets:

1. `accepted_2007_to_2018Q4.csv`
2. `rejected_2007_to_2018Q4.csv`

The complete datasets contain millions of loan applications.  
For computational feasibility, the project will use:

- 250,000 Accepted Applications
- 250,000 Rejected Applications
- Total = 500,000 Applications

No cleaning or feature engineering is performed in Experiment 1.
That will be performed in Experiment 2.

In [ ]:
!pip install -q duckdb dvc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.1/163.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.2/381

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import os
import json
import time

import pandas as pd
import duckdb

In [ ]:
# Location of original LendingClub datasets

RAW_SOURCE_DIR = Path(
    "/content/drive/MyDrive/dataset"
)

ACCEPTED_SOURCE = (
    RAW_SOURCE_DIR /
    "accepted_2007_to_2018Q4.csv"
)

REJECTED_SOURCE = (
    RAW_SOURCE_DIR /
    "rejected_2007_to_2018Q4.csv"
)


# Project folder

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Loan_Approval_XAI_Project"
)

RAW_PROJECT_DIR = PROJECT_ROOT / "data" / "raw"

DOCS_DIR = PROJECT_ROOT / "docs"

REPORTS_DIR = PROJECT_ROOT / "reports" / "exp1"


# Create folders

RAW_PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DOCS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Project folders created.")

Project folders created.


In [ ]:
print("Checking datasets...\n")

print(
    "Accepted dataset exists:",
    ACCEPTED_SOURCE.exists()
)

print(
    "Rejected dataset exists:",
    REJECTED_SOURCE.exists()
)


if not ACCEPTED_SOURCE.exists():
    raise FileNotFoundError(
        f"Accepted dataset not found at:\n{ACCEPTED_SOURCE}"
    )

if not REJECTED_SOURCE.exists():
    raise FileNotFoundError(
        f"Rejected dataset not found at:\n{REJECTED_SOURCE}"
    )


print("\nBoth datasets found successfully.")

Checking datasets...

Accepted dataset exists: True
Rejected dataset exists: True

Both datasets found successfully.


In [ ]:
def file_size_gb(path):

    size_bytes = path.stat().st_size

    return size_bytes / (1024 ** 3)


accepted_size = file_size_gb(
    ACCEPTED_SOURCE
)

rejected_size = file_size_gb(
    REJECTED_SOURCE
)


print(
    f"Accepted dataset size: "
    f"{accepted_size:.2f} GB"
)

print(
    f"Rejected dataset size: "
    f"{rejected_size:.2f} GB"
)

Accepted dataset size: 1.56 GB
Rejected dataset size: 1.66 GB


In [ ]:
con = duckdb.connect()


def sql_path(path):

    return str(path).replace(
        "'",
        "''"
    )

In [ ]:
print(
    "Counting accepted applications..."
)

start = time.time()


accepted_count = con.execute(
    f"""
    SELECT COUNT(*)

    FROM read_csv(
        '{sql_path(ACCEPTED_SOURCE)}',
        header = true,
        all_varchar = true,
        ignore_errors = true
    )
    """
).fetchone()[0]


end = time.time()


print(
    f"Accepted applications: "
    f"{accepted_count:,}"
)

print(
    f"Time taken: "
    f"{end-start:.2f} seconds"
)

Counting accepted applications...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Accepted applications: 2,260,701
Time taken: 27.76 seconds


In [ ]:
print(
    "Counting rejected applications..."
)

start = time.time()


rejected_count = con.execute(
    f"""
    SELECT COUNT(*)

    FROM read_csv(
        '{sql_path(REJECTED_SOURCE)}',
        header = true,
        all_varchar = true,
        ignore_errors = true
    )
    """
).fetchone()[0]


end = time.time()


print(
    f"Rejected applications: "
    f"{rejected_count:,}"
)

print(
    f"Time taken: "
    f"{end-start:.2f} seconds"
)

Counting rejected applications...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rejected applications: 27,648,741
Time taken: 24.59 seconds


In [ ]:
total_applications = (
    accepted_count +
    rejected_count
)


print("=" * 50)

print("ORIGINAL DATASET SUMMARY")

print("=" * 50)

print(
    f"Accepted applications : "
    f"{accepted_count:,}"
)

print(
    f"Rejected applications : "
    f"{rejected_count:,}"
)

print(
    f"Total applications    : "
    f"{total_applications:,}"
)

print("=" * 50)

ORIGINAL DATASET SUMMARY
Accepted applications : 2,260,701
Rejected applications : 27,648,741
Total applications    : 29,909,442


In [ ]:
accepted_structure = con.execute(
    f"""
    DESCRIBE

    SELECT *

    FROM read_csv(
        '{sql_path(ACCEPTED_SOURCE)}',
        header = true,
        all_varchar = true,
        ignore_errors = true
    )
    """
).df()


rejected_structure = con.execute(
    f"""
    DESCRIBE

    SELECT *

    FROM read_csv(
        '{sql_path(REJECTED_SOURCE)}',
        header = true,
        all_varchar = true,
        ignore_errors = true
    )
    """
).df()


accepted_columns = (
    accepted_structure["column_name"]
    .tolist()
)

rejected_columns = (
    rejected_structure["column_name"]
    .tolist()
)


print(
    "Accepted dataset columns:",
    len(accepted_columns)
)

print(
    "Rejected dataset columns:",
    len(rejected_columns)
)

In [ ]:
accepted_structure = con.execute(
    f"""
    DESCRIBE

    SELECT *

    FROM read_csv(
        '{sql_path(ACCEPTED_SOURCE)}',
        header = true,
        all_varchar = true,
        ignore_errors = true
    )
    """
).df()


rejected_structure = con.execute(
    f"""
    DESCRIBE

    SELECT *

    FROM read_csv(
        '{sql_path(REJECTED_SOURCE)}',
        header = true,
        all_varchar = true,
        ignore_errors = true
    )
    """
).df()


accepted_columns = (
    accepted_structure["column_name"]
    .tolist()
)

rejected_columns = (
    rejected_structure["column_name"]
    .tolist()
)


print(
    "Accepted dataset columns:",
    len(accepted_columns)
)

print(
    "Rejected dataset columns:",
    len(rejected_columns)
)

Accepted dataset columns: 151
Rejected dataset columns: 9


In [ ]:
print("ACCEPTED DATASET COLUMNS")

print("-" * 60)

for column in accepted_columns:

    print(column)


print("\n\nREJECTED DATASET COLUMNS")

print("-" * 60)

for column in rejected_columns:

    print(column)

ACCEPTED DATASET COLUMNS
------------------------------------------------------------
id
member_id
loan_amnt
funded_amnt
funded_amnt_inv
term
int_rate
installment
grade
sub_grade
emp_title
emp_length
home_ownership
annual_inc
verification_status
issue_d
loan_status
pymnt_plan
url
desc
purpose
title
zip_code
addr_state
dti
delinq_2yrs
earliest_cr_line
fico_range_low
fico_range_high
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
open_acc
pub_rec
revol_bal
revol_util
total_acc
initial_list_status
out_prncp
out_prncp_inv
total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_d
last_pymnt_amnt
next_pymnt_d
last_credit_pull_d
last_fico_range_high
last_fico_range_low
collections_12_mths_ex_med
mths_since_last_major_derog
policy_code
application_type
annual_inc_joint
dti_joint
verification_status_joint
acc_now_delinq
tot_coll_amt
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total

In [ ]:
accepted_preview = con.execute(
    f"""
    SELECT *

    FROM read_csv(
        '{sql_path(ACCEPTED_SOURCE)}',
        header = true,
        all_varchar = true,
        ignore_errors = true
    )

    LIMIT 5
    """
).df()


display(
    accepted_preview
)

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,None,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,None,None,Cash,N,None,None,None,None,None,None
1,68355089,None,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,None,None,Cash,N,None,None,None,None,None,None
2,68341763,None,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,None,None,Cash,N,None,None,None,None,None,None
3,66310712,None,35000.0,35000.0,35000.0,60 months,14.85,829.9,C,C5,...,None,None,Cash,N,None,None,None,None,None,None
4,68476807,None,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,None,None,Cash,N,None,None,None,None,None,None


In [ ]:
rejected_preview = con.execute(
    f"""
    SELECT *

    FROM read_csv(
        '{sql_path(REJECTED_SOURCE)}',
        header = true,
        all_varchar = true,
        ignore_errors = true
    )

    LIMIT 5
    """
).df()


display(
    rejected_preview
)

,Amount Requested,Application Date,Loan Title,Risk_Score,Debt-To-Income Ratio,Zip Code,State,Employment Length,Policy Code
0,1000.0,2007-05-26,Wedding Covered but No Honeymoon,693.0,10%,481xx,NM,4 years,0.0
1,1000.0,2007-05-26,Consolidating Debt,703.0,10%,010xx,MA,< 1 year,0.0
2,11000.0,2007-05-27,Want to consolidate my debt,715.0,10%,212xx,MD,1 year,0.0
3,6000.0,2007-05-27,waksman,698.0,38.64%,017xx,MA,< 1 year,0.0
4,1500.0,2007-05-27,mdrigo,509.0,9.43%,209xx,MD,< 1 year,0.0


In [ ]:
N_ACCEPTED = 250_000

N_REJECTED = 250_000

RANDOM_SEED = 42


print(
    "Accepted records to use:",
    f"{N_ACCEPTED:,}"
)

print(
    "Rejected records to use:",
    f"{N_REJECTED:,}"
)

print(
    "Total:",
    f"{N_ACCEPTED + N_REJECTED:,}"
)

Accepted records to use: 250,000
Rejected records to use: 250,000
Total: 500,000


In [ ]:
ACCEPTED_SAMPLE_PATH = (
    RAW_PROJECT_DIR /
    "accepted_sample_250k.csv"
)


con.execute(
    f"""
    COPY
    (
        SELECT *

        FROM read_csv(
            '{sql_path(ACCEPTED_SOURCE)}',
            header = true,
            all_varchar = true,
            ignore_errors = true
        )

        USING SAMPLE
        reservoir(
            {N_ACCEPTED} ROWS
        )

        REPEATABLE(
            {RANDOM_SEED}
        )
    )

    TO
    '{sql_path(ACCEPTED_SAMPLE_PATH)}'

    WITH
    (
        HEADER,
        DELIMITER ','
    );
    """
)


print(
    "Accepted sample created:"
)

print(
    ACCEPTED_SAMPLE_PATH
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Accepted sample created:
/content/drive/MyDrive/Loan_Approval_XAI_Project/data/raw/accepted_sample_250k.csv


In [ ]:
REJECTED_SAMPLE_PATH = (
    RAW_PROJECT_DIR /
    "rejected_sample_250k.csv"
)


con.execute(
    f"""
    COPY
    (
        SELECT *

        FROM read_csv(
            '{sql_path(REJECTED_SOURCE)}',
            header = true,
            all_varchar = true,
            ignore_errors = true
        )

        USING SAMPLE
        reservoir(
            {N_REJECTED} ROWS
        )

        REPEATABLE(
            {RANDOM_SEED}
        )
    )

    TO
    '{sql_path(REJECTED_SAMPLE_PATH)}'

    WITH
    (
        HEADER,
        DELIMITER ','
    );
    """
)


print(
    "Rejected sample created:"
)

print(
    REJECTED_SAMPLE_PATH
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rejected sample created:
/content/drive/MyDrive/Loan_Approval_XAI_Project/data/raw/rejected_sample_250k.csv


In [ ]:
accepted_sample_count = (
    con.execute(
        f"""
        SELECT COUNT(*)

        FROM read_csv(
            '{sql_path(ACCEPTED_SAMPLE_PATH)}',
            header = true,
            all_varchar = true
        )
        """
    )
    .fetchone()[0]
)


rejected_sample_count = (
    con.execute(
        f"""
        SELECT COUNT(*)

        FROM read_csv(
            '{sql_path(REJECTED_SAMPLE_PATH)}',
            header = true,
            all_varchar = true
        )
        """
    )
    .fetchone()[0]
)


print(
    "Accepted sample:",
    f"{accepted_sample_count:,}"
)

print(
    "Rejected sample:",
    f"{rejected_sample_count:,}"
)

print(
    "Total:",
    f"{accepted_sample_count + rejected_sample_count:,}"
)

Accepted sample: 250,000
Rejected sample: 250,000
Total: 500,000


In [ ]:
class_distribution = pd.DataFrame(
    {
        "Loan Decision": [
            "Accepted",
            "Rejected"
        ],

        "Number of Records": [
            accepted_sample_count,
            rejected_sample_count
        ]
    }
)


class_distribution[
    "Percentage"
] = (

    class_distribution[
        "Number of Records"
    ]

    /

    class_distribution[
        "Number of Records"
    ].sum()

    * 100
)


display(
    class_distribution
)

,Loan Decision,Number of Records,Percentage
0,Accepted,250000,50.0
1,Rejected,250000,50.0


In [ ]:
dataset_manifest = pd.DataFrame(
    [
        {
            "dataset":
                "Accepted Loans",

            "source_file":
                ACCEPTED_SOURCE.name,

            "original_rows":
                accepted_count,

            "original_columns":
                len(accepted_columns),

            "project_rows":
                accepted_sample_count,

            "sample_file":
                ACCEPTED_SAMPLE_PATH.name,

            "sampling_method":
                "Reservoir Random Sampling",

            "random_seed":
                RANDOM_SEED
        },

        {
            "dataset":
                "Rejected Loans",

            "source_file":
                REJECTED_SOURCE.name,

            "original_rows":
                rejected_count,

            "original_columns":
                len(rejected_columns),

            "project_rows":
                rejected_sample_count,

            "sample_file":
                REJECTED_SAMPLE_PATH.name,

            "sampling_method":
                "Reservoir Random Sampling",

            "random_seed":
                RANDOM_SEED
        }
    ]
)


display(
    dataset_manifest
)


MANIFEST_PATH = (
    REPORTS_DIR /
    "dataset_manifest.csv"
)


dataset_manifest.to_csv(
    MANIFEST_PATH,
    index=False
)


print(
    "Manifest saved:",
    MANIFEST_PATH
)

,dataset,source_file,original_rows,original_columns,project_rows,sample_file,sampling_method,random_seed
0,Accepted Loans,accepted_2007_to_2018Q4.csv,2260701,151,250000,accepted_sample_250k.csv,Reservoir Random Sampling,42
1,Rejected Loans,rejected_2007_to_2018Q4.csv,27648741,9,250000,rejected_sample_250k.csv,Reservoir Random Sampling,42


Manifest saved: /content/drive/MyDrive/Loan_Approval_XAI_Project/reports/exp1/dataset_manifest.csv


# Problem Statement

Financial institutions receive a large number of loan applications and must determine whether each application should be accepted or rejected. Traditional lending decisions generally rely on predefined eligibility rules, credit/risk scores, debt obligations, requested loan amount, employment information, and other financial characteristics.

The objective of this project is to develop an **Explainable Machine Learning System for Loan Application Approval Prediction** using historical LendingClub loan application data from 2007 to 2018 Q4.

Two datasets are used:

1. `accepted_2007_to_2018Q4.csv` containing historical accepted or funded loan applications.
2. `rejected_2007_to_2018Q4.csv` containing historical rejected loan applications.

The final machine learning task will be formulated as a binary classification problem:

- Accepted = 1
- Rejected = 0

The complete LendingClub datasets contain millions of records. Therefore, for computational feasibility in Google Colab, a representative project dataset of 500,000 applications will be used:

- 250,000 Accepted applications
- 250,000 Rejected applications

The accepted and rejected datasets have different schemas. Therefore, compatible application-time features will be identified, cleaned, and standardized during Experiment 2.

Only features that would reasonably be available when the applicant submits the loan application will be considered for machine learning. Information generated after the loan is approved, such as repayment status, recoveries, outstanding balance, and payment history, will be excluded to prevent target leakage.

Machine learning models such as Logistic Regression, Decision Tree, Random Forest, and gradient-boosting models will later be evaluated.

Explainable AI techniques such as SHAP and LIME will be applied to explain why the model predicts an application as accepted or rejected.

The proposed system is intended as an educational and analytical model of historical LendingClub approval decisions and is not intended to replace an actual regulated financial institution's loan underwriting process.

# Benchmark of Existing Solutions

Different approaches can be used for loan application assessment.

| Approach | Advantages | Limitations |
|---|---|---|
| Rule-Based System | Simple and easy to understand | Fixed rules may not capture complex patterns |
| Logistic Regression | Simple and interpretable | Limited ability to learn complex nonlinear relationships |
| Decision Tree | Easy to visualize and explain | Can overfit |
| Random Forest | Handles complex nonlinear relationships | Less directly interpretable |
| Gradient Boosting / XGBoost | Often provides strong predictive performance | More computationally complex |

## Proposed Improvements

The proposed project improves on basic loan prediction systems by:

1. Using both accepted and rejected historical loan applications.
2. Preventing target leakage by selecting only application-time features.
3. Handling schema differences between accepted and rejected datasets.
4. Comparing multiple machine learning models.
5. Evaluating models using multiple classification metrics instead of accuracy alone.
6. Applying SHAP and LIME for model explainability.
7. Using Git, DVC and MLflow for reproducibility and experiment tracking.

# Success Metrics

The project is a binary classification problem.

## Target

- Accepted = 1
- Rejected = 0

## Main Evaluation Metrics

The following metrics will be used:

- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC
- Balanced Accuracy
- Confusion Matrix

## Initial Target Performance

The initial project targets are:

- ROC-AUC >= 0.85
- F1-Score >= 0.80
- Recall >= 0.85

These are initial benchmark targets and may be reviewed after baseline model training.

Accuracy will not be considered alone because other metrics such as precision, recall, F1-score and ROC-AUC provide a more complete evaluation of the classifier.

In [ ]:
dataset_description = f"""

DATASET DESCRIPTION

Dataset:
LendingClub Accepted and Rejected
Loan Applications

Period:
2007 to 2018 Q4


1. ACCEPTED DATASET

File:
{ACCEPTED_SOURCE.name}

Original Rows:
{accepted_count:,}

Original Columns:
{len(accepted_columns)}

Project Sample:
{accepted_sample_count:,}


2. REJECTED DATASET

File:
{REJECTED_SOURCE.name}

Original Rows:
{rejected_count:,}

Original Columns:
{len(rejected_columns)}

Project Sample:
{rejected_sample_count:,}


3. COMPLETE ORIGINAL DATA

Total Applications:
{total_applications:,}


4. PROJECT DATA

Accepted:
{accepted_sample_count:,}

Rejected:
{rejected_sample_count:,}

Total:
{accepted_sample_count + rejected_sample_count:,}


Sampling Method:
Reservoir Random Sampling

Random Seed:
{RANDOM_SEED}


IMPORTANT:

The accepted and rejected datasets have
different schemas.

Schema alignment, missing-value handling,
duplicate handling and feature engineering
will therefore be performed in Experiment 2.

"""


print(
    dataset_description
)



DATASET DESCRIPTION

Dataset:
LendingClub Accepted and Rejected
Loan Applications

Period:
2007 to 2018 Q4


1. ACCEPTED DATASET

File:
accepted_2007_to_2018Q4.csv

Original Rows:
2,260,701

Original Columns:
151

Project Sample:
250,000


2. REJECTED DATASET

File:
rejected_2007_to_2018Q4.csv

Original Rows:
27,648,741

Original Columns:
9

Project Sample:
250,000


3. COMPLETE ORIGINAL DATA

Total Applications:
29,909,442


4. PROJECT DATA

Accepted:
250,000

Rejected:
250,000

Total:
500,000


Sampling Method:
Reservoir Random Sampling

Random Seed:
42


IMPORTANT:

The accepted and rejected datasets have
different schemas.

Schema alignment, missing-value handling,
duplicate handling and feature engineering
will therefore be performed in Experiment 2.




In [ ]:
DATASET_DESCRIPTION_PATH = (
    DOCS_DIR /
    "Dataset_Description.txt"
)


with open(
    DATASET_DESCRIPTION_PATH,
    "w"
) as file:

    file.write(
        dataset_description
    )


print(
    "Dataset description saved:"
)

print(
    DATASET_DESCRIPTION_PATH
)

Dataset description saved:
/content/drive/MyDrive/Loan_Approval_XAI_Project/docs/Dataset_Description.txt


In [ ]:
%cd /content/drive/MyDrive/Loan_Approval_XAI_Project

import os


if not os.path.exists(".git"):

    !git init

else:

    print(
        "Git repository already initialized."
    )

/content/drive/MyDrive/Loan_Approval_XAI_Project
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/Loan_Approval_XAI_Project/.git/


In [ ]:
%cd /content/drive/MyDrive/Loan_Approval_XAI_Project


if not os.path.exists(".dvc"):

    !dvc init

else:

    print(
        "DVC already initialized."
    )

/content/drive/MyDrive/Loan_Approval_XAI_Project
Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>


In [ ]:
%cd /content/drive/MyDrive/Loan_Approval_XAI_Project

!dvc add data/raw/accepted_sample_250k.csv

/content/drive/MyDrive/Loan_Approval_XAI_Project
⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Adding data/raw/accepted_sample_250k.csv to cache:   0% 0/1 [00:00<?, ?file/s]
  0% 0/1 [00:00<?, ?file/s{'info': ''}]                                       
100% 1/1 [00:02<00:00,  2.08s/file{'info': ''}]
                                               
  0% 0/1 [00:00<?, ?files/s]
  0% 0/1 [00:00<?, ?files/s{'info': ''}]
100% 1/1 [00:01<00:00,  1.54s/files{'info': ''}]
Adding...: 100% 1/1 [00:05<00:00,  5.74s/file{'info': ''}]

To track the changes with git, run:

	git add data/raw/accepted_sample_250k.csv.dvc data/raw/.gitignore

To enable auto staging, run:

	dvc config core.autostage true


In [ ]:
%cd /content/drive/MyDrive/Loan_Approval_XAI_Project

!dvc add data/raw/rejected_sample_250k.csv

/content/drive/MyDrive/Loan_Approval_XAI_Project
⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Adding data/raw/rejected_sample_250k.csv to cache:   0% 0/1 [00:00<?, ?file/s]
  0% 0/1 [00:00<?, ?file/s{'info': ''}]                                       
100% 1/1 [00:00<00:00,  8.07file/s{'info': ''}]
                                               
  0% 0/1 [00:00<?, ?files/s]
  0% 0/1 [00:00<?, ?files/s{'info': ''}]
100% 1/1 [00:00<00:00,  7.54files/s{'info': ''}]
Adding...: 100% 1/1 [00:00<00:00,  1.96file/s{'info': ''}]

To track the changes with git, run:

	git add data/raw/.gitignore data/raw/rejected_sample_250k.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [ ]:
!ls -lh data/raw/

total 184M
-rw------- 1 root root 169M Sep  3 16:04 accepted_sample_250k.csv
-rw------- 1 root root  109 Sep  3 16:04 accepted_sample_250k.csv.dvc
-rw------- 1 root root  16M Sep  3 16:04 rejected_sample_250k.csv
-rw------- 1 root root  108 Sep  3 16:04 rejected_sample_250k.csv.dvc


In [ ]:
!git status

On branch master

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .dvc/.gitignore
	new file:   .dvc/config
	new file:   .dvcignore

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/
	docs/
	reports/



In [ ]:
!git config user.name "Loan Approval Project"

!git config user.email "ansh24comp@student.mes.ac.in"

In [ ]:
!git add .

!git status

On branch master

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .dvc/.gitignore
	new file:   .dvc/config
	new file:   .dvcignore
	new file:   data/raw/.gitignore
	new file:   data/raw/accepted_sample_250k.csv.dvc
	new file:   data/raw/rejected_sample_250k.csv.dvc
	new file:   docs/Dataset_Description.txt
	new file:   reports/exp1/dataset_manifest.csv



In [ ]:
!git commit -m "Experiment 1 - Case study and dataset preparation"

[master (root-commit) 4ea0551] Experiment 1 - Case study and dataset preparation
 8 files changed, 97 insertions(+)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore
 create mode 100644 data/raw/.gitignore
 create mode 100644 data/raw/accepted_sample_250k.csv.dvc
 create mode 100644 data/raw/rejected_sample_250k.csv.dvc
 create mode 100644 docs/Dataset_Description.txt
 create mode 100644 reports/exp1/dataset_manifest.csv


In [ ]:
print("=" * 65)

print("EXPERIMENT 1 COMPLETED")

print("=" * 65)


print(
    "\nOriginal Accepted Applications :",
    f"{accepted_count:,}"
)

print(
    "Original Rejected Applications :",
    f"{rejected_count:,}"
)

print(
    "Original Total Applications    :",
    f"{total_applications:,}"
)


print(
    "\nProject Accepted Records :",
    f"{accepted_sample_count:,}"
)

print(
    "Project Rejected Records :",
    f"{rejected_sample_count:,}"
)

print(
    "Project Total Records    :",
    f"{accepted_sample_count + rejected_sample_count:,}"
)


print(
    "\nSampling Seed:",
    RANDOM_SEED
)


print(
    "\nGit initialized."
)

print(
    "DVC initialized."
)

print(
    "Raw project datasets tracked with DVC."
)

print("=" * 65)

EXPERIMENT 1 COMPLETED

Original Accepted Applications : 2,260,701
Original Rejected Applications : 27,648,741
Original Total Applications    : 29,909,442

Project Accepted Records : 250,000
Project Rejected Records : 250,000
Project Total Records    : 500,000

Sampling Seed: 42

Git initialized.
DVC initialized.
Raw project datasets tracked with DVC.


# Conclusion

In Experiment 1, the real-world problem was defined as **Loan Application Approval Prediction** using historical LendingClub data.

The accepted and rejected loan application datasets were successfully identified and analyzed to determine their sizes, number of records, and schemas.

Since the complete dataset contains millions of loan applications, a reproducible balanced sample consisting of:

- 250,000 Accepted applications
- 250,000 Rejected applications

was selected, resulting in a total project dataset of **500,000 loan applications**.

The original datasets were not modified.

The problem statement, benchmark of existing solutions, dataset description, and initial success metrics were documented.

Git was initialized for source-code and metadata versioning, while DVC was used to track the large raw project datasets.



In [1]:
from pathlib import Path
import os
import json
import pandas as pd

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Applied Data Science Project"
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
DOCS_DIR = PROJECT_ROOT / "docs"
REPORT_DIR = PROJECT_ROOT / "reports" / "exp1"

ACCEPTED_SAMPLE = RAW_DIR / "accepted_sample_250k.csv"
REJECTED_SAMPLE = RAW_DIR / "rejected_sample_250k.csv"

DOCS_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)

print(
    "Accepted sample exists:",
    ACCEPTED_SAMPLE.exists()
)

print(
    "Rejected sample exists:",
    REJECTED_SAMPLE.exists()
)

Project: /content/drive/MyDrive/Applied Data Science Project
Accepted sample exists: False
Rejected sample exists: False


In [7]:
from pathlib import Path

DATASET_DIR = Path(
    "/content/drive/MyDrive/dataset"
)

ACCEPTED_SOURCE = (
    DATASET_DIR /
    "accepted_2007_to_2018Q4.csv"
)

REJECTED_SOURCE = (
    DATASET_DIR /
    "rejected_2007_to_2018Q4.csv"
)

print("Accepted original dataset exists:", ACCEPTED_SOURCE.exists())
print("Rejected original dataset exists:", REJECTED_SOURCE.exists())

print("\nAccepted path:")
print(ACCEPTED_SOURCE)

print("\nRejected path:")
print(REJECTED_SOURCE)

Accepted original dataset exists: False
Rejected original dataset exists: False

Accepted path:
/content/drive/MyDrive/dataset/accepted_2007_to_2018Q4.csv

Rejected path:
/content/drive/MyDrive/dataset/rejected_2007_to_2018Q4.csv


In [8]:
%cd "/content/drive/MyDrive/Applied Data Science Project"

# Add the DVC-generated gitignore
!git add data/raw/.gitignore

# Add remaining Experiment 1 files
!git add .dvc/config
!git add data/raw/*.dvc
!git add docs/
!git add reports/exp1/
!git add ADS_EXP_1.ipynb

# Commit Experiment 1
!git commit -m "Complete Experiment 1 - case study framing dataset preparation and DVC versioning"

# Show latest commits
!git log --oneline -3

/content/drive/MyDrive/Applied Data Science Project
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
